In [27]:
import os
import pandas as pd
import numpy as np

# Read the main file and keep only columns that exist in all available CSV files
# This avoids errors when some files are missing columns such as ParkingSpaces or CloseDate.
df = pd.read_csv("filtered_crmls.csv")

candidate_paths = ["filtered_crmls.csv"]
crmls_dir = "CRMLSSold"
if os.path.exists(crmls_dir):
    for filename in sorted(os.listdir(crmls_dir)):
        if filename.lower().endswith(".csv"):
            candidate_paths.append(os.path.join(crmls_dir, filename))

if len(candidate_paths) > 1:
    columns_by_file = [set(pd.read_csv(path).columns) for path in candidate_paths]
    common_columns = set.intersection(*columns_by_file)
    if common_columns:
        df = df[[col for col in df.columns if col in common_columns]].copy()

print(df.shape)


C:\Users\saman\AppData\Local\Temp\ipykernel_3340\2968753406.py:17: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  columns_by_file = [set(pd.read_csv(path).columns) for path in candidate_paths]


(83495, 78)


In [28]:
# keep all columns except ListPrice and OriginalPrice, then remove rows with a lot of missing values
columns_to_drop = [col for col in df.columns if col.lower() in {"listprice", "originalprice"}]
df = df.drop(columns=columns_to_drop, errors="ignore").copy()

missing_ratio = df.isna().mean(axis=1)
df = df.loc[missing_ratio <= 0.5].copy()

print(f"Rows before filtering: {len(missing_ratio)}")
print(f"Rows after filtering: {df.shape[0]}")
print(df.shape)


Rows before filtering: 83495
Rows after filtering: 83495
(83495, 77)


In [29]:
# remove unrealistic rows such as extreme parking counts or properties with no bedrooms/bathrooms
parking_col = "ParkingSpaces" if "ParkingSpaces" in df.columns else None
bedrooms_col = "BedroomsTotal" if "BedroomsTotal" in df.columns else None
bathrooms_col = "BathroomsTotalInteger" if "BathroomsTotalInteger" in df.columns else None

for col in [parking_col, bedrooms_col, bathrooms_col]:
    if col is not None:
        df[col] = pd.to_numeric(df[col], errors="coerce")

mask = pd.Series(True, index=df.index)

if parking_col is not None:
    mask &= df[parking_col].between(0, 20)
if bedrooms_col is not None:
    mask &= df[bedrooms_col].gt(0)
if bathrooms_col is not None:
    mask &= df[bathrooms_col].gt(0)

df = df.loc[mask].copy()

print(f"Rows after outlier filtering: {df.shape[0]}")
print(df.shape)


Rows after outlier filtering: 83444
(83444, 77)


In [30]:
#identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

numeric_cols, categorical_cols



C:\Users\saman\AppData\Local\Temp\ipykernel_3340\101513429.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()


(['OriginalListPrice',
  'ListingKey',
  'ClosePrice',
  'Latitude',
  'Longitude',
  'LivingArea',
  'DaysOnMarket',
  'FireplacesTotal',
  'AboveGradeFinishedArea',
  'ListingKeyNumeric',
  'TaxAnnualAmount',
  'ParkingTotal',
  'LotSizeAcres',
  'YearBuilt',
  'StreetNumberNumeric',
  'BathroomsTotalInteger',
  'TaxYear',
  'BuildingAreaTotal',
  'BedroomsTotal',
  'ElementarySchoolDistrict',
  'BelowGradeFinishedArea',
  'BusinessType',
  'CoveredSpaces',
  'Stories',
  'LotSizeArea',
  'MainLevelBedrooms',
  'GarageSpaces',
  'AssociationFee',
  'LotSizeSquareFeet',
  'MiddleOrJuniorSchoolDistrict'],
 ['BuyerAgentAOR',
  'ListAgentAOR',
  'Flooring',
  'ViewYN',
  'WaterfrontYN',
  'BasementYN',
  'PoolPrivateYN',
  'ListAgentEmail',
  'CloseDate',
  'ListAgentFirstName',
  'ListAgentLastName',
  'UnparsedAddress',
  'PropertyType',
  'ListOfficeName',
  'BuyerOfficeName',
  'CoListOfficeName',
  'ListAgentFullName',
  'CoListAgentFirstName',
  'CoListAgentLastName',
  'BuyerAgent

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

safe_categorical = [
    "City",
    "CountyOrParish",
    "PostalCode",
    "PropertyType",
    "PropertySubType",
    "StateOrProvince",
    "MLSAreaMajor",
    "ElementarySchoolDistrict",
    "MiddleOrJuniorSchoolDistrict",
    "HighSchoolDistrict"
]

if "CloseDate" not in df.columns:
    df = pd.read_csv("filtered_crmls.csv")

selected_columns = [c for c in safe_categorical + numeric_cols + ["CloseDate"] if c in df.columns]
df = df[selected_columns].copy()
df = df.loc[:, ~df.columns.duplicated()]

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = encoder.fit_transform(df[safe_categorical])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(safe_categorical))

In [32]:
df[numeric_cols].isna().sum().sort_values(ascending=False)


AboveGradeFinishedArea          83444
FireplacesTotal                 83444
TaxAnnualAmount                 83444
BusinessType                    83444
ElementarySchoolDistrict        83444
TaxYear                         83444
CoveredSpaces                   83444
MiddleOrJuniorSchoolDistrict    83444
BelowGradeFinishedArea          82841
BuildingAreaTotal               77897
MainLevelBedrooms               32555
AssociationFee                  24012
Stories                          8697
GarageSpaces                     3199
LotSizeAcres                     1442
LotSizeSquareFeet                1442
LotSizeArea                      1436
OriginalListPrice                 175
StreetNumberNumeric               113
YearBuilt                          44
LivingArea                         39
Longitude                          11
Latitude                           11
ParkingTotal                        0
DaysOnMarket                        0
ClosePrice                          0
ListingKey  

In [33]:
# concatenate the encoded categorical columns with the numeric columns
numeric_cols = [
    "ClosePrice",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LivingArea",
    "Latitude",
    "Longitude",
    "YearBuilt"
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

df = df.reset_index(drop=True).copy()

df_encoded = pd.concat([
    df[numeric_cols].reset_index(drop=True),
    encoded_df.reset_index(drop=True)
], axis=1)


In [34]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_numeric = scaler.fit_transform(df_encoded[numeric_cols])
df_encoded[numeric_cols] = scaled_numeric



In [35]:
df = df.reset_index(drop=True).copy()

df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
df["Month"] = df["CloseDate"].dt.to_period("M")

latest_month = df["Month"].max()

test_mask = df["Month"] == latest_month

if len(df) != len(df_encoded):
    df_encoded = df_encoded.iloc[:len(df)].copy()

# Use the same index alignment for train/test split
common_index = df.index

df_test = df_encoded.loc[test_mask.to_numpy()].copy()
df_train = df_encoded.loc[(~test_mask).to_numpy()].copy()

# Export cleaned data for future use
output_path = "cleaned_preprocessed.csv"
df_encoded.to_csv(output_path, index=False)
print(f"Saved cleaned data to {output_path}")
print(df_train.shape)
print(df_test.shape)


Saved cleaned data to cleaned_preprocessed.csv
(71428, 4203)
(12016, 4203)
